# Exercise 1.1: Your First AI Weather Forecast

**Level**: Beginner | **Time**: 15 min | **GPU**: Optional (faster with GPU)

Run a global 5-day weather forecast using NVIDIA's FourCastNet3 in under 10 lines of Python.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

**What you'll learn:**
- Earth2Studio's 4-component pattern: **Model + Data + IO + Run**
- How pretrained model weights auto-download
- What FourCastNet3 predicts (73 atmospheric channels)
- How to visualize global weather maps

## Setup (Colab)

Run this cell first to install dependencies. Takes ~2 minutes.

In [ ]:
# Install Earth2Studio + FCN3 dependencies
# torch-harmonics from PyPI, makani from NVIDIA's GitHub (not on PyPI)
!pip install -q "earth2studio>=0.13.0" torch-harmonics matplotlib cartopy xarray zarr scipy
!pip install -q "makani @ git+https://github.com/NVIDIA/makani.git"

# Verify
import earth2studio
import torch
print(f"earth2studio: {earth2studio.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 1: Run a 5-Day Forecast

Earth2Studio uses a clean 4-component pattern for ALL models:

| Component | What it does | Example |
|-----------|-------------|---------|
| **Model** | Steps the atmosphere forward in time | FCN3, Atlas, DLWP |
| **Data** | Provides initial conditions (real atmosphere) | GFS, ERA5/ARCO |
| **IO** | Stores the output | ZarrBackend, NetCDF |
| **Run** | Orchestrates the forecast loop | deterministic, ensemble |

Once you learn this pattern, you can swap any component independently.

In [ ]:
from earth2studio.models.px import FCN3
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio import run

# Load pretrained FourCastNet3 (~500MB download on first run)
print("Loading FCN3 model (first run downloads weights)...")
package = FCN3.load_default_package()
model = FCN3.load_model(package)
print("Model loaded!")

In [ ]:
# Run the forecast!
# - GFS: real-time NOAA atmospheric analysis
# - nsteps=20: 20 × 6hr = 120hr = 5 days
# - ZarrBackend: chunked array storage

print("Running 5-day forecast (20 steps × 6hr)...")
io = run.deterministic(
    time=["2025-06-01T00:00:00"],  # Try changing to a recent date!
    nsteps=20,
    model=model,
    data=GFS(),
    io=ZarrBackend("forecast.zarr")
)
print("Forecast complete!")

## Part 2: Explore the Output

The forecast is stored in Zarr format — a chunked, cloud-native array format.
We use `xarray` to read it, which gives us labeled dimensions.

In [ ]:
import xarray as xr

ds = xr.open_zarr("forecast.zarr")

print(f"Dimensions: {dict(ds.dims)}")
print(f"\nVariables ({len(ds.coords['variable'])} total):")
for v in sorted(ds.coords['variable'].values):
    print(f"  {v}")

In [ ]:
# Spatial resolution
lats = ds.coords['lat'].values
lons = ds.coords['lon'].values
print(f"Grid: {len(lats)} lat × {len(lons)} lon")
print(f"Resolution: {abs(lats[1]-lats[0]):.2f}° ≈ {abs(lats[1]-lats[0])*111:.0f} km at equator")
print(f"\nTime steps: {len(ds.coords['lead_time'])}")
print(f"Lead times: {ds.coords['lead_time'].values}")

## Part 3: Visualize — 500 hPa Geopotential Height

This is the classic "weather map" that meteorologists use:
- **Troughs** (low z500) = stormy weather
- **Ridges** (high z500) = fair weather
- The wavy pattern is the **Rossby wave** train that drives midlatitude weather

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print("Cartopy not available — using basic projection")

# Select z500 at T+48h (step index 8)
step = 8  # T+48h
z500 = ds.sel(variable="z500").isel(time=0, lead_time=step).values.squeeze()
z500_dam = z500 / (9.81 * 10)  # Convert to decameters

if HAS_CARTOPY:
    fig, ax = plt.subplots(figsize=(14, 8), subplot_kw={'projection': ccrs.Robinson()})
    cf = ax.contourf(lons, lats, z500_dam, levels=np.arange(480, 600, 4),
                     cmap='RdYlBu_r', transform=ccrs.PlateCarree())
    cs = ax.contour(lons, lats, z500_dam, levels=np.arange(480, 600, 8),
                    colors='black', linewidths=0.5, transform=ccrs.PlateCarree())
    ax.clabel(cs, inline=True, fontsize=8, fmt='%.0f')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.set_global()
    plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.05, label='Height (dam)')
else:
    fig, ax = plt.subplots(figsize=(14, 6))
    cf = ax.contourf(lons, lats, z500_dam, levels=np.arange(480, 600, 4), cmap='RdYlBu_r')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    plt.colorbar(cf, ax=ax, label='Height (dam)')

ax.set_title(f'500 hPa Geopotential Height — T+{step*6}h Forecast\n'
             f'FourCastNet3 | Init: {str(ds.coords["time"].values[0])[:10]}', fontsize=14)
plt.tight_layout()
plt.show()

## Part 4: 2m Temperature Evolution

Watch how the global temperature field evolves over 5 days.

In [ ]:
steps = [0, 4, 8, 12, 16, 20]  # Every 24 hours
steps = [s for s in steps if s < len(ds.coords['lead_time'])]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, s in enumerate(steps):
    t2m = ds.sel(variable="t2m").isel(time=0, lead_time=s).values.squeeze()
    t2m_c = t2m - 273.15  # Kelvin → Celsius
    
    cf = axes[i].contourf(lons, lats, t2m_c, levels=np.arange(-40, 45, 5), cmap='RdYlBu_r')
    axes[i].set_title(f'T+{s*6}h (Day {s*6//24})')

fig.suptitle('2m Temperature Evolution (°C) — FCN3 Forecast', fontsize=16, y=1.02)
fig.colorbar(cf, ax=axes, orientation='horizontal', pad=0.08, label='°C', shrink=0.8)
plt.tight_layout()
plt.show()

## Part 5: Regional Zoom — Your City

Earth2Studio uses **0-360° longitude**. To convert:
- New York (74°W) → `360 - 74 = 286`
- London (0°) → `0`
- Tokyo (140°E) → `140`

In [ ]:
# CONUS zoom
lat_range = (25, 50)
lon_range = (235, 295)  # ~125°W to ~65°W

lat_mask = (lats >= lat_range[0]) & (lats <= lat_range[1])
lon_mask = (lons >= lon_range[0]) & (lons <= lon_range[1])

t2m_48h = ds.sel(variable="t2m").isel(time=0, lead_time=8).values.squeeze()
regional = (t2m_48h[np.ix_(lat_mask, lon_mask)]) - 273.15

fig, ax = plt.subplots(figsize=(12, 8))
cf = ax.contourf(lons[lon_mask], lats[lat_mask], regional, levels=30, cmap='RdYlBu_r')
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title('2m Temperature (°C) — Continental US — T+48h', fontsize=14)
plt.colorbar(cf, ax=ax, label='Temperature (°C)')
plt.tight_layout()
plt.show()

## Part 6: 10m Wind Speed (Derived Variable)

FCN3 predicts U and V wind components separately. Wind speed = √(u² + v²).

In [ ]:
u10 = ds.sel(variable="u10m").isel(time=0, lead_time=8).values.squeeze()
v10 = ds.sel(variable="v10m").isel(time=0, lead_time=8).values.squeeze()
wspd = np.sqrt(u10**2 + v10**2)

fig, ax = plt.subplots(figsize=(14, 6))
cf = ax.contourf(lons, lats, wspd, levels=np.arange(0, 25, 1), cmap='YlOrRd')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('10m Wind Speed (m/s) — T+48h — FCN3', fontsize=14)
plt.colorbar(cf, ax=ax, label='Wind Speed (m/s)')
plt.tight_layout()
plt.show()

---

## Exercises

Try these modifications in the cells above:

1. **Change the date** to today. Does the forecast change? (It should!)
2. **Increase nsteps to 40** (10-day forecast). How much longer does it take?
3. **Plot sea-level pressure** (`msl`) instead of z500. Where are the highs and lows?
4. **Zoom to your region** — change `lat_range` and `lon_range` in Part 5
5. **Plot the difference** between T+0 and T+120h for temperature. Where changes most?

---

**Next**: Exercise 1.2 (Visualization) and 1.3 (Model Comparison)